In [1]:
import os, sys, math, random, copy, time, warnings
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from scipy import interpolate
from tqdm.auto import tqdm
warnings.filterwarnings('ignore')

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW

SEED = 42
random.seed(SEED); np.random.seed(SEED)
torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark     = False
torch.use_deterministic_algorithms(True, warn_only=True)
os.environ['PYTHONHASHSEED'] = str(SEED)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {DEVICE}')
if DEVICE.type == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name(0)}')


Device: cuda
GPU: Tesla T4


In [2]:
class CFG:
    TRAIN_PATH   = '/kaggle/input/datasets/fda137/spectral-graffiti/spectral_graffiti_headley_recovered.csv'
    CKPT_PATH    = '/kaggle/working/sanp_best.pt'
    EMA_CKPT     = '/kaggle/working/sanp_ema.pt'
    T_MIN, T_MAX = 1.0, 100.0
    VAL_FRAC     = 0.02
    GP_SCALES    = [2.0, 4.0, 8.0, 15.0, 25.0]
    GP_NOISE     = 1e-3
    N_GP         = len(GP_SCALES)
    FOURIER_DIM  = 64
    FOURIER_MAX  = 50.0
    D_MODEL      = 128
    N_HEADS      = 4
    N_ENC        = 3
    N_DEC        = 3
    D_FF         = 256
    DROPOUT      = 0.10
    BATCH_SIZE   = 128
    EPOCHS       = 100
    LR           = 3e-4
    MIN_LR_FRAC  = 0.05
    WEIGHT_DECAY = 5e-4
    GRAD_CLIP    = 1.0
    EMA_DECAY    = 0.9995
    HUBER_DELTA  = 0.5
    PATIENCE     = 12
    NUM_WORKERS  = 4
    CTX_NOISE_STD = 0.002
    MIN_CTX      = 20
    EDGE_LOSS_W  = 1.0


In [3]:
print('Loading data...')
t0 = time.time()
df = pd.read_csv(
    CFG.TRAIN_PATH,
    dtype={'Sample_ID':np.int32,'Time_ms':np.int16,
           'Is_Context':np.int8,'Value':np.float32},
)
print(f'  {len(df):,} rows  {time.time()-t0:.1f}s')

def build_sample_dict(df):
    samples = {}
    for sid, grp in tqdm(df.groupby('Sample_ID'), desc='Indexing', leave=False):
        grp = grp.sort_values('Time_ms')
        samples[int(sid)] = {
            'times' : grp['Time_ms'].values.astype(np.float32),
            'values': grp['Value'].values.astype(np.float32),
            'is_ctx': grp['Is_Context'].values.astype(bool),
        }
    return samples

all_samples = build_sample_dict(df)
all_ids     = np.array(sorted(all_samples.keys()))
_rng = np.random.RandomState(SEED)
_rng.shuffle(all_ids)
n_val   = max(400, int(len(all_ids) * CFG.VAL_FRAC))
val_ids = all_ids[:n_val].copy()
fit_ids = all_ids[n_val:].copy()
print(f'  Train: {len(fit_ids):,}  Val: {len(val_ids):,}')


Loading data...
  8,000,000 rows  2.4s


Indexing:   0%|          | 0/80000 [00:00<?, ?it/s]

  Train: 78,400  Val: 1,600


In [4]:
def rbf_kernel(t1, t2, l):
    return np.exp(-0.5 * ((t1[:,None] - t2[None,:]) / l)**2)

def compute_features(ctx_t, ctx_v, tgt_t,
                     scales=CFG.GP_SCALES, noise=CFG.GP_NOISE):
    n   = len(ctx_t)
    out = np.empty((len(scales)+1, len(tgt_t)), dtype=np.float32)
    for i, l in enumerate(scales):
        K_cc = rbf_kernel(ctx_t, ctx_t, l) + noise*np.eye(n)
        K_tc = rbf_kernel(tgt_t, ctx_t, l)
        try:    out[i] = K_tc @ np.linalg.solve(K_cc, ctx_v)
        except: out[i] = np.interp(tgt_t, ctx_t, ctx_v)
    out[-1] = (interpolate.CubicSpline(ctx_t, ctx_v, bc_type='natural')(tgt_t)
               if n >= 4 else np.interp(tgt_t, ctx_t, ctx_v))
    return out

def gp_posterior_std(ctx_t, ctx_v, tgt_t, l=8.0, noise=CFG.GP_NOISE):
    n    = len(ctx_t)
    K_cc = rbf_kernel(ctx_t, ctx_t, l) + noise*np.eye(n)
    K_tc = rbf_kernel(tgt_t, ctx_t, l)
    try:
        alpha = np.linalg.solve(K_cc, K_tc.T)
        var   = np.maximum(1.0 - np.einsum('ij,ij->j', K_tc.T, alpha), 1e-6)
    except:
        var   = np.ones(len(tgt_t), dtype=np.float32)
    return np.sqrt(var).astype(np.float32)

def normalize_time(t):
    return 2.0*(np.asarray(t, np.float32)-CFG.T_MIN)/(CFG.T_MAX-CFG.T_MIN)-1.0


In [5]:
def worker_init_fn(wid):
    np.random.seed(SEED+wid); random.seed(SEED+wid)

class SpectralDataset(Dataset):
    def __init__(self, ids, sample_dict, train=True):
        self.ids=ids; self.data=sample_dict; self.train=train
    def __len__(self): return len(self.ids)

    def __getitem__(self, idx):
        sid = int(self.ids[idx])
        s   = self.data[sid]
        m   = s['is_ctx']
        t_c = s['times'][m].astype(np.float64)
        v_c = s['values'][m].astype(np.float64)
        t_t = s['times'][~m].astype(np.float64)
        v_t = s['values'][~m].astype(np.float64)

        if self.train and CFG.CTX_NOISE_STD > 0:
            v_c = v_c + np.random.randn(len(v_c)) * CFG.CTX_NOISE_STD

        mu_c  = float(v_c.mean())
        std_c = float(max(v_c.std(), 1e-4))
        cv_n  = ((v_c - mu_c)/std_c).astype(np.float32)
        tv_n  = ((v_t - mu_c)/std_c).astype(np.float32)

        feats    = compute_features(t_c, v_c, t_t)
        f_norm   = ((feats - mu_c)/std_c).astype(np.float32)
        gp_sigma = np.clip(gp_posterior_std(t_c, v_c, t_t), 0.01, 2.0)
        spline_n = f_norm[-1]
        delta_v  = tv_n - spline_n

        t_t_int = t_t.astype(int)
        loss_w  = np.where((t_t_int<=2)|(t_t_int>=99),
                            CFG.EDGE_LOSS_W, 1.0).astype(np.float32)

        return dict(
            ctx_t    = torch.from_numpy(normalize_time(t_c)),
            ctx_v    = torch.from_numpy(cv_n),
            tgt_t    = torch.from_numpy(normalize_time(t_t)),
            tgt_v    = torch.from_numpy(tv_n),
            delta_v  = torch.from_numpy(delta_v),
            gp_feats = torch.from_numpy(f_norm.T),
            spline   = torch.from_numpy(spline_n),
            gp_sigma = torch.from_numpy(gp_sigma),
            loss_w   = torch.from_numpy(loss_w),
            mu_c     = torch.tensor(mu_c,  dtype=torch.float32),
            std_c    = torch.tensor(std_c, dtype=torch.float32),
            sid      = sid,
        )

def collate_fn(batch):
    def stack(k): return torch.stack([b[k] for b in batch])
    return dict(
        ctx_t    = stack('ctx_t'),    ctx_v   = stack('ctx_v'),
        tgt_t    = stack('tgt_t'),    tgt_v   = stack('tgt_v'),
        delta_v  = stack('delta_v'),  spline  = stack('spline'),
        gp_feats = stack('gp_feats'), gp_sigma= stack('gp_sigma'),
        loss_w   = stack('loss_w'),   mu_c    = stack('mu_c'),
        std_c    = stack('std_c'),    sids    = [b['sid'] for b in batch],
    )

g = torch.Generator(); g.manual_seed(SEED)
train_ds = SpectralDataset(fit_ids, all_samples, train=True)
val_ds   = SpectralDataset(val_ids, all_samples, train=False)
train_dl = DataLoader(train_ds, CFG.BATCH_SIZE, shuffle=True,
    collate_fn=collate_fn, num_workers=CFG.NUM_WORKERS,
    pin_memory=True, persistent_workers=True,
    worker_init_fn=worker_init_fn, generator=g)
val_dl = DataLoader(val_ds, CFG.BATCH_SIZE, shuffle=False,
    collate_fn=collate_fn, num_workers=CFG.NUM_WORKERS,
    pin_memory=True, persistent_workers=True,
    worker_init_fn=worker_init_fn)

_b = next(iter(train_dl))
print(f'ctx={_b["ctx_t"].shape}  tgt={_b["tgt_t"].shape}  gp={_b["gp_feats"].shape}')
print(f'delta std={_b["delta_v"].std():.4f}  tgt std={_b["tgt_v"].std():.4f}')
assert _b["delta_v"].std() < _b["tgt_v"].std(), "Spline anchor not reducing residual variance"
print('Dataset ready!!')


ctx=torch.Size([128, 20])  tgt=torch.Size([128, 80])  gp=torch.Size([128, 80, 6])
delta std=0.2385  tgt std=1.0047
Dataset ready!!


In [6]:
class LogSpacedFourier(nn.Module):
    def __init__(self):
        super().__init__()
        freqs = torch.logspace(0, math.log10(math.pi*CFG.FOURIER_MAX), CFG.FOURIER_DIM)
        self.register_buffer('freqs', freqs)
        self.out_dim = 2*CFG.FOURIER_DIM
    def forward(self, t):
        x = t.unsqueeze(-1)*self.freqs
        return torch.cat([torch.sin(x), torch.cos(x)], dim=-1)

class GatedResFFN(nn.Module):
    def __init__(self, D, d_ff, dropout):
        super().__init__()
        self.norm = nn.LayerNorm(D)
        self.ffn  = nn.Sequential(nn.Linear(D,d_ff), nn.GELU(),
                                   nn.Dropout(dropout), nn.Linear(d_ff,D))
        self.gate = nn.Sequential(nn.Linear(D,D), nn.Sigmoid())
    def forward(self, x):
        return x + self.gate(x)*self.ffn(self.norm(x))

class DistBiasedCrossAttn(nn.Module):
    def __init__(self, D, n_heads, dropout):
        super().__init__()
        self.attn    = nn.MultiheadAttention(D, n_heads, dropout=dropout, batch_first=True)
        self.scale   = nn.Parameter(torch.tensor(0.5))
        self.norm_q  = nn.LayerNorm(D)
        self.norm_kv = nn.LayerNorm(D)
    def forward(self, q, kv, tgt_t, ctx_t):
        B, Nt = q.shape[:2]
        Nc = kv.shape[1]; n_h = self.attn.num_heads
        dist = (tgt_t.unsqueeze(2) - ctx_t.unsqueeze(1)).abs()
        bias = (-dist/self.scale.abs().clamp(min=1e-3))
        bias = bias.unsqueeze(1).expand(-1,n_h,-1,-1).reshape(B*n_h,Nt,Nc)
        out, _ = self.attn(self.norm_q(q), self.norm_kv(kv), self.norm_kv(kv),
                           attn_mask=bias)
        return q + out

class DecoderBlock(nn.Module):
    def __init__(self, D, n_heads, d_ff, dropout):
        super().__init__()
        self.cross = DistBiasedCrossAttn(D, n_heads, dropout)
        self.norm  = nn.LayerNorm(D)
        self.sa    = nn.MultiheadAttention(D, n_heads, dropout=dropout, batch_first=True)
        self.ffn   = GatedResFFN(D, d_ff, dropout)
    def forward(self, q, kv, tgt_t, ctx_t):
        q = self.cross(q, kv, tgt_t, ctx_t)
        r = self.norm(q); a,_ = self.sa(r,r,r); q = q + a
        return self.ffn(q)

class SANP(nn.Module):
    def __init__(self):
        super().__init__()
        D  = CFG.D_MODEL
        td = 2*CFG.FOURIER_DIM
        N6 = CFG.N_GP+1
        self.fourier  = LogSpacedFourier()
        self.spec_tok = nn.Parameter(torch.randn(1,1,D)*0.02)
        self.ctx_proj = nn.Sequential(nn.Linear(td+1,D), nn.LayerNorm(D),
                                       nn.GELU(), nn.Linear(D,D))
        enc_layer = nn.TransformerEncoderLayer(D, CFG.N_HEADS, CFG.D_FF, CFG.DROPOUT,
                        activation='gelu', batch_first=True, norm_first=True)
        self.encoder  = nn.TransformerEncoder(enc_layer, CFG.N_ENC,
                                               enable_nested_tensor=False)
        self.tgt_proj = nn.Sequential(nn.Linear(td+N6,D), nn.LayerNorm(D),
                                       nn.GELU(), nn.Linear(D,D))
        self.ctx_gate = nn.Linear(D*2, D)
        self.decoder  = nn.ModuleList([
            DecoderBlock(D, CFG.N_HEADS, CFG.D_FF, CFG.DROPOUT)
            for _ in range(CFG.N_DEC)])
        self.delta = nn.Sequential(
            nn.LayerNorm(D), nn.Linear(D,CFG.D_FF), nn.GELU(),
            nn.Dropout(CFG.DROPOUT),
            nn.Linear(CFG.D_FF, CFG.D_FF//2), nn.GELU(),
            nn.Linear(CFG.D_FF//2, 1))
        nn.init.normal_(self.delta[-1].weight, std=0.001)
        nn.init.zeros_(self.delta[-1].bias)

    def forward(self, ctx_t, ctx_v, tgt_t, gp_feats, spline):
        B, Nt = tgt_t.shape
        ctx_emb = self.fourier(ctx_t)
        tgt_emb = self.fourier(tgt_t)
        ctx_tok = self.ctx_proj(torch.cat([ctx_emb, ctx_v.unsqueeze(-1)], -1))
        spec    = self.spec_tok.expand(B,-1,-1)
        ctx_seq = torch.cat([spec, ctx_tok], dim=1)
        ctx_repr= self.encoder(ctx_seq)
        spec_out= ctx_repr[:,0,:]
        ctx_repr= ctx_repr[:,1:,:]
        q       = self.tgt_proj(torch.cat([tgt_emb, gp_feats], -1))
        ctx_exp = spec_out.unsqueeze(1).expand(-1,Nt,-1)
        gate    = torch.sigmoid(self.ctx_gate(torch.cat([q, ctx_exp], -1)))
        q       = q + gate*ctx_exp
        for blk in self.decoder:
            q = blk(q, ctx_repr, tgt_t, ctx_t)
        return spline + self.delta(q).squeeze(-1)

model = SANP().to(DEVICE)
n_p = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Parameters: {n_p:,}')
with torch.no_grad():
    _out = model(_b['ctx_t'][:4].to(DEVICE), _b['ctx_v'][:4].to(DEVICE),
                 _b['tgt_t'][:4].to(DEVICE),
                 gp_feats=_b['gp_feats'][:4].to(DEVICE),
                 spline=_b['spline'][:4].to(DEVICE))
print(f'Output: {_out.shape}  range [{_out.min():.3f}, {_out.max():.3f}]')


Parameters: 1,210,884
Output: torch.Size([4, 80])  range [-2.888, 2.481]


In [7]:
class EMA:
    def __init__(self, model, decay=CFG.EMA_DECAY):
        self.decay  = decay
        self.shadow = copy.deepcopy(model)
        self.shadow.eval()
        for p in self.shadow.parameters(): p.requires_grad_(False)
    @torch.no_grad()
    def update(self, model):
        for sp,mp in zip(self.shadow.parameters(), model.parameters()):
            sp.data.lerp_(mp.data, 1-self.decay)
    def save(self, path):
        torch.save(self.shadow.state_dict(), path)

ema       = EMA(model)
optimizer = AdamW(model.parameters(), lr=CFG.LR,
                  weight_decay=CFG.WEIGHT_DECAY, betas=(0.9,0.98), eps=1e-8)
scheduler = torch.optim.lr_scheduler.CosineAnnealingWarmRestarts(
    optimizer, T_0=20, T_mult=2, eta_min=CFG.LR*CFG.MIN_LR_FRAC)
scaler    = torch.cuda.amp.GradScaler(enabled=(DEVICE.type=='cuda'))
gstep     = 0

def weighted_huber(pred, target, sigma, loss_w):
    r = pred - target
    h = torch.where(r.abs()<CFG.HUBER_DELTA,
                    0.5*r**2,
                    CFG.HUBER_DELTA*(r.abs()-0.5*CFG.HUBER_DELTA))
    return (sigma.detach().clamp(0.5,2.0)*loss_w*h).mean()

def run_epoch(loader, mdl, train=True):
    global gstep
    mdl.train(train)
    tot_loss=tot_n=0
    with torch.set_grad_enabled(train):
        for batch in tqdm(loader, desc='train' if train else 'val',
                          leave=False, mininterval=2):
            ct  = batch['ctx_t'].to(DEVICE, non_blocking=True)
            cv  = batch['ctx_v'].to(DEVICE, non_blocking=True)
            tt  = batch['tgt_t'].to(DEVICE, non_blocking=True)
            tv  = batch['tgt_v'].to(DEVICE, non_blocking=True)
            dv  = batch['delta_v'].to(DEVICE, non_blocking=True)
            gf  = batch['gp_feats'].to(DEVICE, non_blocking=True)
            sp  = batch['spline'].to(DEVICE, non_blocking=True)
            sig = batch['gp_sigma'].to(DEVICE, non_blocking=True)
            lw  = batch['loss_w'].to(DEVICE, non_blocking=True)

            with torch.amp.autocast('cuda', enabled=(train and DEVICE.type=='cuda')):
                pred       = mdl(ct, cv, tt, gp_feats=gf, spline=sp)
                delta_pred = pred - sp
                loss       = weighted_huber(delta_pred, dv, sig, lw)

            if not torch.isfinite(loss):
                if train: optimizer.zero_grad(set_to_none=True)
                continue

            if train:
                optimizer.zero_grad(set_to_none=True)
                scaler.scale(loss).backward()
                scaler.unscale_(optimizer)
                nn.utils.clip_grad_norm_(model.parameters(), CFG.GRAD_CLIP)
                scaler.step(optimizer); scaler.update()
                scheduler.step(); ema.update(model); gstep+=1

            with torch.no_grad():
                mse = ((pred.detach() - tv)**2).mean()
            tot_loss += mse.item()*tv.numel()
            tot_n    += tv.numel()
    return tot_loss/max(tot_n,1)

history={'train':[],'val':[]}
best_val=float('inf')
no_improve=0

for epoch in range(1, CFG.EPOCHS+1):
    tr = run_epoch(train_dl, model,      train=True)
    vl = run_epoch(val_dl,   ema.shadow, train=False)
    history['train'].append(tr); history['val'].append(vl)
    flag=''
    if vl < best_val:
        best_val=vl; no_improve=0
        torch.save(model.state_dict(), CFG.CKPT_PATH)
        ema.save(CFG.EMA_CKPT); flag='  ★'
    else:
        no_improve+=1
    print(f'Epoch {epoch:3d}/{CFG.EPOCHS}  Train={tr:.6f}  Val={vl:.6f}{flag}')
    if no_improve>=CFG.PATIENCE:
        print(f'Early stop at epoch {epoch}')
        break

print(f'\nBest val MSE: {best_val:.6f}')


train:   0%|          | 0/613 [00:00<?, ?it/s]

val:   0%|          | 0/13 [00:00<?, ?it/s]

Epoch   1/100  Train=0.044250  Val=0.066785  ★


train:   0%|          | 0/613 [00:00<?, ?it/s]

val:   0%|          | 0/13 [00:00<?, ?it/s]

...

val:   0%|          | 0/13 [00:00<?, ?it/s]

Epoch  33/100  Train=0.027445  Val=0.035307


train:   0%|          | 0/613 [00:00<?, ?it/s]

val:   0%|          | 0/13 [00:00<?, ?it/s]

Epoch  34/100  Train=0.029069  Val=0.035287
Early stop at epoch 34

Best val MSE: 0.033860


In [8]:
fig, ax = plt.subplots(figsize=(10,4))
ep = range(1, len(history['train'])+1)
ax.plot(ep, history['train'], label='Train', color='steelblue', lw=1.5)
ax.plot(ep, history['val'],   label='Val (EMA)',  color='coral',     lw=2.0)
ax.set_xlabel('Epoch'); ax.set_title('MSE over epochs')
ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout(); plt.savefig('/kaggle/working/loss.png', dpi=100)
plt.show(); plt.close()

model.load_state_dict(torch.load(CFG.EMA_CKPT, map_location=DEVICE, weights_only=True))
model.eval()

_t  = torch.randn(1,20).to(DEVICE); _v  = torch.randn(1,20).to(DEVICE)
_tt = torch.randn(1,80).to(DEVICE); _gf = torch.randn(1,80,CFG.N_GP+1).to(DEVICE)
_sp = torch.randn(1,80).to(DEVICE)
with torch.no_grad():
    o1 = model(_t,_v,_tt,gp_feats=_gf,spline=_sp)
    o2 = model(_t,_v,_tt,gp_feats=_gf,spline=_sp)
assert torch.allclose(o1,o2,atol=1e-5), 'Non-deterministic!'
print(f'Determinism: max diff = {(o1-o2).abs().max().item():.2e}')
print(f'Best val MSE: {best_val:.6f}')


Determinism: max diff = 0.00e+00
Best val MSE: 0.033860


In [9]:
def predict_sample(s, mdl):
    m   = s['is_ctx']
    t_c = s['times'][m].astype(np.float64)
    v_c = s['values'][m].astype(np.float64)
    t_t = s['times'][~m].astype(np.float64)
    if len(t_c) == 0:
        return np.full(len(t_t), 0.5, dtype=np.float32)
    mu_c  = float(v_c.mean())
    std_c = float(max(v_c.std(), 1e-4))
    feats    = compute_features(t_c, v_c, t_t)
    f_norm   = ((feats - mu_c)/std_c).astype(np.float32)
    spline_n = f_norm[-1]
    cv_n     = ((v_c - mu_c)/std_c).astype(np.float32)
    ct = torch.from_numpy(normalize_time(t_c)).unsqueeze(0).to(DEVICE)
    cv = torch.from_numpy(cv_n).unsqueeze(0).to(DEVICE)
    tt = torch.from_numpy(normalize_time(t_t)).unsqueeze(0).to(DEVICE)
    gf = torch.from_numpy(f_norm.T).unsqueeze(0).to(DEVICE)
    sp = torch.from_numpy(spline_n).unsqueeze(0).to(DEVICE)
    with torch.no_grad():
        pred_n = mdl(ct, cv, tt, gp_feats=gf, spline=sp)
    return pred_n.squeeze(0).cpu().float().numpy() * std_c + mu_c

_s0 = all_samples[int(val_ids[0])]
_p1 = predict_sample(_s0, model)
_p2 = predict_sample(_s0, model)
assert np.allclose(_p1,_p2,atol=1e-6), 'Inference non-deterministic!'
print(f'Inference determinism: max diff = {np.abs(_p1-_p2).max():.2e}')

errs=[]
for sid in val_ids[:200]:
    s=all_samples[sid]; p=predict_sample(s,model)
    errs.extend(((p-s['values'][~s['is_ctx']])**2).tolist())
print(f'Val raw MSE : {np.mean(errs):.6f}')
print(f'Val raw RMSE: {np.mean(errs)**0.5:.6f}')


Inference determinism: max diff = 0.00e+00
Val raw MSE : 0.001893
Val raw RMSE: 0.043504


# For Round 1 submission

In [ ]:
'''
model.eval()
rows = []
for sid in tqdm(all_ids, desc='Generating submission'):
    s     = all_samples[sid]
    mask  = s['is_ctx']
    t_all = s['times']
    v_all = s['values']
    t_tgt = t_all[~mask]
    preds = predict_sample(s, model)
    pred_map = {int(t): float(p) for t, p in zip(t_tgt, preds)}
    for t, v, is_c in zip(t_all, v_all, mask):
        t = int(t)
        val = float(v) if is_c else pred_map[t]
        rows.append((sid, t, round(val, 6)))
sub = pd.DataFrame(rows, columns=['Sample_ID','Time_ms','Predicted_Value'])
sub = sub.sort_values(['Sample_ID','Time_ms']).reset_index(drop=True)
sub.to_csv('/kaggle/working/submission_round1.csv', index=False)
print(f'Saved: {len(sub):,} rows  (expected {len(all_ids)*100:,})')
print(sub.head(6))
'''

# **FOR ROUND 2 SUBMISSION**

In [10]:
TEST_PATH = '/kaggle/input/datasets/nikitaa1/test-dataset/test_features_spectral - test_features_spectral.csv'
test_df = pd.read_csv(
    TEST_PATH,
    dtype={
        'Sample_ID': np.int32,
        'Time_ms': np.int16,
        'Is_Context': np.int8,
        'Value': np.float32
    }
)

test_samples = build_sample_dict(test_df)
test_ids = np.array(sorted(test_samples.keys()))

model.load_state_dict(torch.load(CFG.EMA_CKPT, map_location=DEVICE, weights_only=True))
model.eval()

rows2 = []

for sid in tqdm(test_ids, desc='Round-2'):
    s = test_samples[sid]
    mask = s['is_ctx']
    t_all = s['times']
    v_all = s['values']

    t_tgt = t_all[~mask]
    preds = predict_sample(s, model)

    pred_map = {int(t): float(p) for t, p in zip(t_tgt, preds)}

    for t, v, is_c in zip(t_all, v_all, mask):
        t = int(t)
        val = float(v) if is_c else pred_map[t]
        rows2.append((sid, t, round(val, 6)))

sub2 = pd.DataFrame(rows2, columns=['Sample_ID', 'Time_ms', 'Predicted_Value'])

sub2.sort_values(['Sample_ID', 'Time_ms']).to_csv(
    '/kaggle/working/finalsubmission.csv',
    index=False
)

print(f'Round-2: {len(sub2):,} rows saved  (expected {len(test_ids)*100:,})')

Indexing:   0%|          | 0/10000 [00:00<?, ?it/s]

Round-2:   0%|          | 0/10000 [00:00<?, ?it/s]

Round-2: 1,000,000 rows saved  (expected 1,000,000)
